# Multiple Loggers, Handlers, Hierarchy, and Rotation

> **Goal:** understand how a larger application can give each part its own logger while sharing sensible configuration.

## Explain it like I am 5

Imagine a school. Each classroom writes its own diary, but all classroom diaries can send important notes to the principal's office. Logger names use dots to make a family: `school`, `school.math`, `school.math.quiz`.

This notebook preserves the original `module1`/`module2` example and expands it into production-friendly patterns.

## 1. The four moving pieces

| Piece | Simple job |
|---|---|
| `Logger` | creates and routes a record |
| `LogRecord` | the event package |
| `Handler` | sends it to console/file/etc. |
| `Formatter` | decides how it looks |

One logger can have several handlers. Several child loggers can share ancestor handlers through **propagation**.

## 2. Preserve the original two-logger example

`module1` accepts `DEBUG+`; `module2` accepts `WARNING+`. The shared root handler prints accepted records.

In [1]:
import logging

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True,
)

logger1 = logging.getLogger("module1")
logger1.setLevel(logging.DEBUG)

logger2 = logging.getLogger("module2")
logger2.setLevel(logging.WARNING)

logger1.debug("This is debug message for module1")
logger2.info("This INFO is filtered out")
logger2.warning("This is a warning message for module2")
logger2.error("This is an error message")

2026-09-07 02:59:00 - module1 - DEBUG - This is debug message for module1


2026-09-07 02:59:00 - module2 - WARNING - This is a warning message for module2


2026-09-07 02:59:00 - module2 - ERROR - This is an error message


## 3. Logger hierarchy

Dots create ancestry:

```text
shop
├── shop.catalog
└── shop.checkout
    └── shop.checkout.payment
```

A child's `propagate=True` (the default) sends accepted records upward to ancestor handlers. Logger levels are subtle: a child with level `NOTSET` inherits an effective level from its ancestors, but ancestor logger levels are not reapplied during propagation; handler levels still filter records.

In [2]:
root_logger = logging.getLogger()
shop_logger = logging.getLogger("shop")
payment_logger = logging.getLogger("shop.checkout.payment")

shop_logger.setLevel(logging.INFO)
payment_logger.setLevel(logging.NOTSET)  # Inherits effective level from shop/root family.

print("payment parent name:", payment_logger.parent.name)
print("payment effective level:", logging.getLevelName(payment_logger.getEffectiveLevel()))
print("propagate:", payment_logger.propagate)
payment_logger.info("Payment module is ready")

2026-09-07 02:59:00 - shop.checkout.payment - INFO - Payment module is ready


payment parent name: shop
payment effective level: INFO
propagate: True


## 4. Why duplicate log lines happen

If a child has its own handler **and** propagates to an ancestor that also has a handler, the same record may be printed twice.

Two valid designs:

1. **Central design:** handlers live high in the tree; children propagate.
2. **Isolated design:** a child owns handlers and sets `propagate = False`.

Do not blindly add handlers each time a module or notebook cell runs.

In [3]:
import sys

isolated_logger = logging.getLogger("demo.isolated")
isolated_logger.handlers.clear()
isolated_logger.setLevel(logging.DEBUG)
isolated_logger.propagate = False

isolated_console = logging.StreamHandler(sys.stdout)
isolated_console.setFormatter(logging.Formatter("ISOLATED | %(levelname)s | %(message)s"))
isolated_logger.addHandler(isolated_console)
isolated_logger.info("Printed once because propagation is off")

ISOLATED | INFO | Printed once because propagation is off


## 5. Console + file with different levels

This is the most useful practical arrangement:

- console: `INFO+` for people watching now;
- file: `DEBUG+` for later diagnosis.

We write only to a temporary directory, leaving the repository's logs unchanged.

In [4]:
from pathlib import Path
from tempfile import TemporaryDirectory

temporary_logs = TemporaryDirectory()
runtime_dir = Path(temporary_logs.name)
details_path = runtime_dir / "details.log"

app_logger = logging.getLogger("store")
app_logger.handlers.clear()
app_logger.setLevel(logging.DEBUG)
app_logger.propagate = False

console = logging.StreamHandler(sys.stdout)
console.setLevel(logging.INFO)
console.setFormatter(logging.Formatter("CONSOLE | %(levelname)s | %(name)s | %(message)s"))

file_handler = logging.FileHandler(details_path, mode="w", encoding="utf-8")
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(logging.Formatter(
    "%(asctime)s | %(levelname)s | %(name)s | %(funcName)s | %(message)s"
))

app_logger.addHandler(console)
app_logger.addHandler(file_handler)

catalog_logger = logging.getLogger("store.catalog")
checkout_logger = logging.getLogger("store.checkout")

catalog_logger.debug("Loaded 3 products")
checkout_logger.info("Order accepted")
checkout_logger.warning("Payment response was slow")
file_handler.flush()

print("\nFILE CONTENTS")
print(details_path.read_text(encoding="utf-8"))

CONSOLE | INFO | store.checkout | Order accepted


CONSOLE | WARNING | store.checkout | Payment response was slow



FILE CONTENTS
2026-09-07 02:59:00,813 | DEBUG | store.catalog | <module> | Loaded 3 products
2026-09-07 02:59:00,813 | INFO | store.checkout | <module> | Order accepted
2026-09-07 02:59:00,814 | WARNING | store.checkout | <module> | Payment response was slow



## 6. Rotating logs

An endless file is like a diary that grows until it fills the room. `RotatingFileHandler` starts a new file after a size limit.

- `maxBytes`: approximate size limit for each active file.
- `backupCount`: how many older files to retain.

For time-based rotation, use `TimedRotatingFileHandler(when='midnight', backupCount=7)`. Rotation is not a complete retention strategy: decide compression, permissions, collection, and deletion for the real system.

In [5]:
from logging.handlers import RotatingFileHandler

rotating_path = runtime_dir / "rotating.log"
rotation_logger = logging.getLogger("store.rotation_demo")
rotation_logger.handlers.clear()
rotation_logger.setLevel(logging.INFO)
rotation_logger.propagate = False

rotating_handler = RotatingFileHandler(
    rotating_path,
    maxBytes=180,
    backupCount=2,
    encoding="utf-8",
)
rotating_handler.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
rotation_logger.addHandler(rotating_handler)

for number in range(12):
    rotation_logger.info("Event %02d: a deliberately padded demonstration message", number)

rotating_handler.flush()
print(sorted(path.name for path in runtime_dir.glob("rotating.log*")))

['rotating.log', 'rotating.log.1', 'rotating.log.2']


### Rotation edge cases

- Multiple processes writing the same rotating file can race; use a process-safe aggregation design or platform logging service.
- Rotation may occur only when a new record is emitted.
- `backupCount=0` means old backups are not kept/rotation behavior differs by handler.
- Ensure the directory exists before creating a `FileHandler`.
- On Windows, close handlers before trying to move/delete their files.

## 7. Filters: a small gate with custom rules

Levels are not the only filter. A `logging.Filter` can add context or allow/reject records. Here we add a `service` field so the formatter always has it.

In [6]:
class ServiceFilter(logging.Filter):
    def __init__(self, service_name):
        super().__init__()
        self.service_name = service_name

    def filter(self, record):
        record.service = self.service_name
        return True


filtered_handler = logging.StreamHandler(sys.stdout)
filtered_handler.addFilter(ServiceFilter("checkout-api"))
filtered_handler.setFormatter(logging.Formatter(
    "%(levelname)s | service=%(service)s | %(message)s"
))

filtered_logger = logging.getLogger("filter.demo")
filtered_logger.handlers.clear()
filtered_logger.addHandler(filtered_handler)
filtered_logger.setLevel(logging.INFO)
filtered_logger.propagate = False
filtered_logger.info("Health check passed")

INFO | service=checkout-api | Health check passed


## 8. Central configuration with `dictConfig()`

For larger applications, a dictionary keeps formatters, handlers, and loggers in one visible plan. `disable_existing_loggers=False` avoids unexpectedly silencing loggers created by libraries.

In [7]:
from logging.config import dictConfig

dictConfig({
    "version": 1,
    "disable_existing_loggers": False,
    "formatters": {
        "simple": {"format": "%(levelname)s | %(name)s | %(message)s"},
    },
    "handlers": {
        "demo_console": {
            "class": "logging.StreamHandler",
            "level": "INFO",
            "formatter": "simple",
            "stream": "ext://sys.stdout",
        },
    },
    "loggers": {
        "configured_app": {
            "handlers": ["demo_console"],
            "level": "DEBUG",
            "propagate": False,
        },
    },
})

configured_logger = logging.getLogger("configured_app.worker")
configured_logger.debug("Filtered by the INFO console handler")
configured_logger.info("Configured centrally")

INFO | configured_app.worker | Configured centrally


## 9. Exception logging at a boundary

Lower-level code should often raise a meaningful exception. The application boundary—the place that decides what to do—logs it once with context.

In [8]:
def parse_quantity(text):
    quantity = int(text)
    if quantity < 0:
        raise ValueError("quantity cannot be negative")
    return quantity


try:
    parse_quantity("many")
except ValueError:
    configured_logger.exception("Could not parse order quantity")

ERROR | configured_app.worker | Could not parse order quantity
Traceback (most recent call last):
  File "C:\Users\HP\AppData\Local\Temp\ipykernel_34660\2049365390.py", line 9, in <module>
    parse_quantity("many")
    ~~~~~~~~~~~~~~^^^^^^^^
  File "C:\Users\HP\AppData\Local\Temp\ipykernel_34660\2049365390.py", line 2, in parse_quantity
    quantity = int(text)
ValueError: invalid literal for int() with base 10: 'many'


## 10. Performance and `isEnabledFor()`

Lazy arguments avoid string formatting, but Python still evaluates function arguments. Guard genuinely expensive diagnostic work.

In [9]:
def expensive_snapshot():
    return {"items": 1250, "status": "ready"}

if configured_logger.isEnabledFor(logging.DEBUG):
    configured_logger.debug("Snapshot: %s", expensive_snapshot())
else:
    print("Skipped expensive DEBUG snapshot.")

## 11. Existing files as a practical system

The repository's `app.py` uses a named arithmetic logger plus `FileHandler("app1.log")` and `StreamHandler()`. The `logs/logger.py` and `logs/test.py` pair demonstrates shared configuration through an import.

The import pattern works for a lesson, but production modules should import `logging`, call `getLogger(__name__)`, and leave configuration to the entry point. Importing `logging` *from another module* hides where the standard library object came from and makes modules tightly coupled.

In [10]:
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "app.py").exists():
    candidate = NOTEBOOK_DIR / "Complete-Python-Bootcamp-main" / "12-Logging In Python"
    if candidate.exists():
        NOTEBOOK_DIR = candidate

for relative_name in ["app.py", "logs/logger.py", "logs/test.py"]:
    path = NOTEBOOK_DIR / relative_name
    print(relative_name, "->", len(path.read_text(encoding="utf-8").splitlines()), "lines")

for relative_name in ["app.log", "app1.log", "logs/app.log"]:
    path = NOTEBOOK_DIR / relative_name
    first_line = path.read_text(encoding="utf-8").splitlines()[0]
    print(relative_name, "first record:", first_line)

app.py -> 44 lines
logs/logger.py -> 10 lines
logs/test.py -> 8 lines
app.log first record: 2024-06-19 12:25:46-root-DEBUG-This is a debug message
app1.log first record: 2024-06-19 13:34:00 - ArithmethicApp - DEBUG - Subtracting 15 - 10 = 5
logs/app.log first record: 2024-06-19 12:29:57-root-DEBUG-The addition function is called


## 12. Real-world design pattern

```python
# package/payment.py
import logging
logger = logging.getLogger(__name__)

def charge(order_id):
    logger.info("Charging order %s", order_id)

# main.py
import logging
from logging.config import dictConfig

if __name__ == "__main__":
    dictConfig(LOGGING_CONFIG)
    run_application()
```

The module reports events. The application decides where they go. This separation makes libraries reusable and tests quieter.

## 13. Advanced concepts to know next

- **`QueueHandler` / `QueueListener`:** move slow log writing away from worker threads.
- **Structured logging:** emit JSON or key/value records for machines to search.
- **Correlation IDs:** connect records belonging to one request across services.
- **`contextvars`:** carry request context safely in asynchronous code.
- **UTC timestamps:** simplify systems across time zones.
- **Remote collection:** production systems often ship stdout/files to a centralized platform.
- **Testing:** `unittest.assertLogs` or pytest's `caplog` can assert emitted records.

Logging must not become business logic. The program should still behave correctly if logging is unavailable.

## 14. Common mistakes and best practices

| Mistake | Better choice |
|---|---|
| handlers attached in every module | configure centrally |
| child handler plus propagation by accident | centralize, or set `propagate=False` |
| different ad-hoc logger names | use `getLogger(__name__)` |
| one never-ending log file | rotate and define retention |
| catching an error only to log and hide it | recover meaningfully or re-raise |
| personal/secrets in context | redact and minimize |
| logging every loop iteration at `INFO` | choose useful milestones; use `DEBUG` carefully |
| believing logging is free | guard expensive diagnostics |

## 15. Cleanup

Close every notebook-created handler before deleting temporary logs. `logging.shutdown()` exists for process exit, but targeted cleanup is friendlier inside a shared notebook kernel.

In [11]:
for demo_logger in [isolated_logger, app_logger, rotation_logger, filtered_logger]:
    for handler in demo_logger.handlers[:]:
        handler.flush()
        handler.close()
        demo_logger.removeHandler(handler)

temporary_logs.cleanup()
print("Temporary handlers and files cleaned up; existing logs were untouched.")

Temporary handlers and files cleaned up; existing logs were untouched.


# Quick Revision Cheat Sheet

## Hierarchy

```python
logger = logging.getLogger(__name__)  # e.g. shop.payment
logger.info("Paid order %s", order_id)
```

- Dotted names form a family.
- `propagate=True` sends records toward ancestor handlers.
- Child-owned handlers often pair with `propagate=False`.
- Logger and handler levels are separate gates.

## Multiple destinations

```python
logger.setLevel(logging.DEBUG)
console.setLevel(logging.INFO)
file_handler.setLevel(logging.DEBUG)
logger.addHandler(console)
logger.addHandler(file_handler)
```

## Rotation

```python
from logging.handlers import RotatingFileHandler
handler = RotatingFileHandler(
    "app.log", maxBytes=1_000_000, backupCount=5, encoding="utf-8"
)
```

## Golden rules

- Configure in the entry point; use `getLogger(__name__)` elsewhere.
- Avoid duplicate handlers and accidental propagation.
- Log exceptions once at the handling boundary.
- Rotate files, define retention, protect private data, and use structured context.